# EV Charging Stations — Data Understanding & Cleaning

**Dataset:** NDAP source code `8486` — *Electricity Consumption Details of EV Charging Stations of Distribution Companies (DISCOMs)*, Pan-India, monthly, FY 2022-23 & FY 2023-24.

**Goal of this notebook (Step 1):** Thoroughly understand the raw data, then clean and standardise it, and export analysis-ready CSVs. Insight-finding happens in a later notebook using these cleaned exports.

---
### Files in the download
| File | Role |
|---|---|
| `8486_source_data.csv` | **Main dataset** — what DISCOMs reported, original names. *We clean this one.* |
| `NDAP_REPORT_8486.csv` | Same data re-mapped to India's official LGD geography. Splits rows across districts → double-counts consumption, so **not** used for stats. |
| `8486_METADATA.csv` | Data dictionary (variable names, units, descriptions). Reference only. |
| `8486_KEYS.csv` | Empty — note says no geographic keys generated for a Pan-India dataset. |
| `readme.pdf` | Documentation. |

### Cleaning decisions (made explicit so they can be changed)
1. **Fully-empty rows** (a company reported nothing on all 5 measures) → exported separately as "non-reporting", **dropped** from the analysis table. A blank report is not a true zero.
2. **Partial nulls** (some measures missing) → **left as NaN**. Imputing 0 would understate real consumption and bias the statistics.
3. **Company names** → punctuation/spacing variants **collapsed**; field **split** into `DISCOM` + `State`.
4. **City names** → case / whitespace / comma-spacing normalised, OCR mid-word gaps repaired, a leaked footnote removed.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW = '/Users/anahitasaxena/Desktop/EV DA/8486_source_data.csv'   # adjust path if needed
df = pd.read_csv(RAW)
print('Loaded:', df.shape)
df.head()

Loaded: (662, 11)


,Country,srcYear,srcMonth,Name of distribution company,Name of major city,Number of EV Charging Stations excluding Heavy Duty Vehicles charging stations (including EV charging points),Number of PCS specific to Heavy Duty Vehicles (E-bus etc.),Electricity consumed in Electric Vehicle (EV) charging stations excluding heavy duty vehicles Public Charging Station (PCS),Electricity consumed in heavy duty Public Charging Station (PCS) only,Total electricity consumed of Electric Vehicle (EV) charging stations,Additional Info
0,India,2022-23,Aug-2022,SBPDCL,Patna,NaN,NaN,NaN,NaN,NaN,NaN
1,India,2022-23,Dec-2022,"LPDCL, UT of Ladakh",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,India,2023-24,Aug-2023,"HESCOM, Kar.",Hubli,12.0,0.0,12807.0,0.0,0.013,NaN
3,India,2023-24,Jan-2023,"LPDCL, UT of Ladakh",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,India,2022-23,May-2022,"NBPDCL, Bihar",Patna,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Data Understanding

Before changing anything, we profile the raw data: structure, types, missingness, duplicates, and value sanity.

In [2]:
print('Shape:', df.shape)
print('\n--- Dtypes ---')
print(df.dtypes)
print('\n--- Nulls per column ---')
print(df.isnull().sum())

Shape: (662, 11)

--- Dtypes ---
Country                                                                                                                         object
srcYear                                                                                                                         object
srcMonth                                                                                                                        object
Name of distribution company                                                                                                    object
Name of major city                                                                                                              object
Number of EV Charging Stations excluding Heavy Duty Vehicles charging stations (including EV charging points)                  float64
Number of PCS specific to Heavy Duty Vehicles (E-bus etc.)                                                                     float64
Electricity consumed i

In [3]:
# Short, readable column aliases for the 5 measures
MEASURES = {
    'Number of EV Charging Stations excluding Heavy Duty Vehicles charging stations (including EV charging points)': 'ev_stations',
    'Number of PCS specific to Heavy Duty Vehicles (E-bus etc.)': 'hd_stations',
    'Electricity consumed in Electric Vehicle (EV) charging stations excluding heavy duty vehicles Public Charging Station (PCS)': 'ev_kwh',
    'Electricity consumed in heavy duty Public Charging Station (PCS) only': 'hd_kwh',
    'Total electricity consumed of Electric Vehicle (EV) charging stations': 'total_mu',
}
measure_cols = list(MEASURES.values())

In [4]:
# Full duplicates and key-level duplicates
print('Full duplicate rows:', df.duplicated().sum())
key_dims = ['Name of distribution company', 'Name of major city', 'srcYear', 'srcMonth']
print('Duplicates on (company, city, year, month):', df.duplicated(subset=key_dims).sum())

Full duplicate rows: 0
Duplicates on (company, city, year, month): 0


In [5]:
# Temporal coverage
print('Financial years:'); print(df['srcYear'].value_counts(), '\n')
print('Distinct months:', df['srcMonth'].nunique())
print('Country values:', df['Country'].unique())

Financial years:
srcYear
2023-24    401
2022-23    261
Name: count, dtype: int64 

Distinct months: 20
Country values: ['India']


### 2.1 Observations

- **662 rows × 11 columns.** No full duplicates, no key-level duplicates.
- **Missingness:** `Additional Info` is empty in 606/662 rows (drop). City missing in 28 rows. The 5 measures have 71–124 nulls; **71 rows are blank on all five** (non-reporting months).
- **Company names** carry both a DISCOM code and a state, with inconsistent punctuation (`AVVNL, Raj` vs `AVVNL, Raj.`).
- **City names** suffer from case, whitespace, comma-spacing, and OCR mid-word gaps (`Visakhapatna m`), plus one row where a footnote leaked into the city field.
- **Sanity:** no negative values; reported `total_mu` reconciles with the kWh columns (rounding only).

## 3. Cleaning & Standardisation

We build the clean table step by step, keeping the raw `df` untouched and logging every transformation.

In [6]:
clean = df.copy()
clean = clean.rename(columns=MEASURES)
# Drop the near-empty Additional Info column (606/662 null)
clean = clean.drop(columns=['Additional Info'])
print('Columns now:', list(clean.columns))

Columns now: ['Country', 'srcYear', 'srcMonth', 'Name of distribution company', 'Name of major city', 'ev_stations', 'hd_stations', 'ev_kwh', 'hd_kwh', 'total_mu']


### 3.1 Standardise the time columns

In [7]:
# srcYear like '2023-24' (financial year); srcMonth like 'Aug-2023'
# Parse srcMonth temporarily to extract year and month; we do NOT keep a date column.
_parsed = pd.to_datetime(clean['srcMonth'], format='%b-%Y')
clean['calendar_year'] = _parsed.dt.year
clean['calendar_month'] = _parsed.dt.month
clean = clean.rename(columns={'srcYear': 'financial_year'})
clean[['srcMonth', 'calendar_year', 'calendar_month', 'financial_year']].head()

,srcMonth,calendar_year,calendar_month,financial_year
0,Aug-2022,2022,8,2022-23
1,Dec-2022,2022,12,2022-23
2,Aug-2023,2023,8,2023-24
3,Jan-2023,2023,1,2023-24
4,May-2022,2022,5,2022-23


### 3.2 Clean the distribution-company field

Split `Name of distribution company` into a canonical `DISCOM` code and `State`, collapsing punctuation/spacing variants.

**Domain-knowledge correction (manual):** The raw value `UT Chandigarh` is *not* a DISCOM — it is a
Union Territory name with the DISCOM omitted. The generic "text before the comma = DISCOM" rule cannot
detect this (there is no programmatic way to know whether a leading token is a company or a territory),
so we handle it explicitly: such entries get `DISCOM = NaN` and the territory moved to `State`.

In [8]:
STATE_FIX = {'Raj.':'Raj','Kar.':'Kar','Guj.':'Guj','Mah.':'Mah','Ker.':'Ker','Tng.':'Tng'}

# DISCOM spelling variants that are the SAME company (spacing/punctuation differences only).
# Hardcoded and manually verified -- collapses phantom duplicate companies.
DISCOM_FIX = {
    'Tata Power- DDL': 'Tata Power-DDL',                              # stray space after hyphen
    'Noida Power Company Ltd.(NPCL)': 'Noida Power Company Ltd. (NPCL)',  # missing space before (
}

# Entries that are a territory/state ONLY, with no DISCOM. Manually identified (domain knowledge).
# Maps the raw company string -> the State it should become (DISCOM will be set to NaN).
TERRITORY_ONLY = {
    'UT Chandigarh': 'Chandigarh',
}

def split_company(s):
    s = re.sub(r'\s*,\s*', ', ', s.strip())   # normalise comma spacing
    s = re.sub(r'\s+', ' ', s)                  # collapse whitespace

    # Territory-only entries: no DISCOM, value belongs in State
    if s in TERRITORY_ONLY:
        return pd.Series([np.nan, TERRITORY_ONLY[s]])

    if ', ' in s:
        code, state = s.rsplit(', ', 1)
    else:
        code, state = s, ''
    state = STATE_FIX.get(state, state)
    code = DISCOM_FIX.get(code.strip(), code.strip())   # collapse DISCOM spelling variants
    return pd.Series([code.strip(), state.strip()])

clean[['DISCOM', 'State']] = clean['Name of distribution company'].apply(split_company)
print('Raw distinct company strings :', df['Name of distribution company'].nunique())
print('Clean distinct DISCOMs       :', clean['DISCOM'].nunique(), '(excludes NaN)')
print('Rows with DISCOM = NaN (territory-only):', clean['DISCOM'].isna().sum())

Raw distinct company strings : 57
Clean distinct DISCOMs       : 34 (excludes NaN)
Rows with DISCOM = NaN (territory-only): 8


#### Impute missing State for SBPDCL

The DISCOM `SBPDCL` appears in two forms in the raw data: `SBPDCL` (state suffix dropped) and
`SBPDCL, Bihar`. We fill the blank state with **Bihar**. This is *label deduplication*, not
imputation of a measured value — its basis is internal to the dataset:

1. The data itself labels the other 12 SBPDCL rows as `SBPDCL, Bihar`.
2. SBPDCL = *South Bihar Power Distribution Company Ltd.* (the "B" is Bihar).
3. Every SBPDCL row reports **Patna** (capital of Bihar) as its city.

Without this merge, `SBPDCL` and `SBPDCL, Bihar` would count as two phantom companies.

In [9]:
# Show the rows being imputed BEFORE the fill, for transparency
sbpdcl = clean[clean['DISCOM'] == 'SBPDCL']
print('SBPDCL rows and their parsed State (before impute):')
print(sbpdcl['State'].value_counts(dropna=False).to_string())

before_blank = (clean['DISCOM'].eq('SBPDCL') & clean['State'].eq('')).sum()
clean.loc[clean['DISCOM'].eq('SBPDCL') & clean['State'].eq(''), 'State'] = 'Bihar'
print(f'\nImputed State=Bihar for {before_blank} bare-SBPDCL rows.')

print('\nClean distinct (DISCOM,State):', clean.groupby(['DISCOM','State']).ngroups)
sorted(clean['DISCOM'].dropna().unique())

SBPDCL rows and their parsed State (before impute):
State
         12
Bihar    12

Imputed State=Bihar for 12 bare-SBPDCL rows.

Clean distinct (DISCOM,State): 34


['APDCL',
 'APEPDCL',
 'APSPDCL',
 'AVVNL',
 'BESCOM',
 'BRPL',
 'BYPL',
 'Brihan Mumbai Electric Supply Company (BEST)',
 'CESC',
 'CESC Mysore',
 'DGVCL',
 'DHBVN',
 'DVC',
 'GESCOM',
 'HESCOM',
 'JDVVNL',
 'JVVNL',
 'KSEB',
 'LPDCL',
 'MESCOM',
 'MPMKVVCL',
 'MSEDCL',
 'NBPDCL',
 'Noida Power Company Ltd. (NPCL)',
 'PGVCL',
 'SBPDCL',
 'TANGEDCO',
 'TPCODL',
 'TSNPDCL',
 'TSSPDCL',
 'Tata Power-DDL',
 'Torrent Power',
 'UGVCL',
 'UHBVN']

#### Relabel State for DVC (cross-border utility)

`DVC` (Damodar Valley Corporation) is filed under `WB` but physically operates in the Damodar Valley,
serving **Maithon** and **Dhanbad** — both in **Jharkhand**. Its data is *unique* (no other DISCOM
reports these cities), so it is genuine, not a mislabel. Since the `State` column otherwise represents
where consumption physically occurs, we relabel DVC's State to **Jharkhand** for geographic accuracy.

In [10]:
dvc_rows = (clean['DISCOM'] == 'DVC').sum()
clean.loc[clean['DISCOM'] == 'DVC', 'State'] = 'Jharkhand'
print(f'Relabelled State = Jharkhand for {dvc_rows} DVC rows.')

Relabelled State = Jharkhand for 20 DVC rows.


### 3.3 Clean the city field

Four problems: (a) case, (b) whitespace, (c) comma-spacing, (d) OCR mid-word gaps. Plus one leaked footnote (removed). We log every OCR repair so it can be verified.

In [11]:
# Remove the footnote that leaked into the city column (discard it entirely)
FOOTNOTE_FLAG = 'Note:'
leaked = clean['Name of major city'].fillna('').str.startswith(FOOTNOTE_FLAG)
print('Footnote rows removed from city column:', leaked.sum())
clean.loc[leaked, 'Name of major city'] = np.nan

Footnote rows removed from city column: 1


In [12]:
def repair_ocr_gaps(s):
    # join a stray trailing 1-2 letter fragment back to the previous word,
    # but only inside a token (not across commas). e.g. 'visakhapatna m' -> 'visakhapatnam'
    parts = [p.strip() for p in s.split(',')]
    fixed = []
    for p in parts:
        p = re.sub(r'\s+', ' ', p).strip()
        p = re.sub(r'([a-z]{3,}) ([a-z]{1,2})\b', r'\1\2', p)  # 'patna m' -> 'patnam'
        fixed.append(p)
    return ', '.join([x for x in fixed if x])

# Complete, manually-verified list of broken OCR city tokens -> correct spelling (all lowercase).
# Built from a full audit of every multi-word token in the city column.
CITY_FIX = {
    'chikkamagal uru': 'chikkamagaluru',
    'chikkamagalur u': 'chikkamagaluru',
    'mahabubaba d':    'mahabubabad',
    'narmadapura m':   'narmadapuram',
    'kok rajhar':      'kokrajhar',
    'kokr ajhar':      'kokrajhar',
    'rajmahendra varam':  'rajmahendravaram',
    'rajmahendrav aram':  'rajmahendravaram',
    'rajmahendravar am':  'rajmahendravaram',
    'surendranaga r':  'surendranagar',
    'thiruvanantha puram':  'thiruvananthapuram',
    'thiruvananthap uram':  'thiruvananthapuram',
    'vapi- town':      'vapi-town',
    'visakhapatna m':  'visakhapatnam',
}

def clean_city(s):
    if pd.isna(s):
        return s
    s = repair_ocr_gaps(s)
    # RULE: city column is fully lowercase
    tokens = [tok.strip().lower() for tok in s.split(',')]
    # normalise hyphen spacing inside a token: 'vapi- town' -> 'vapi-town'
    tokens = [re.sub(r'\s*-\s*', '-', t) for t in tokens]
    # strip trailing 'etc'/'etc.' stuck to a token: 'vizianagaram etc' -> 'vizianagaram'
    tokens = [re.sub(r'\s+etc\.?$', '', t).strip() for t in tokens]
    # drop any token that is ONLY 'etc'/'etc.'
    tokens = [t for t in tokens if t.rstrip('.') != 'etc']
    # apply the verified spelling fixes
    tokens = [CITY_FIX.get(t, t) for t in tokens]
    return ', '.join(t for t in tokens if t)

clean['city'] = clean['Name of major city'].apply(clean_city)

# Show what the OCR/format repair merged
before = df['Name of major city'].dropna().nunique()
after = clean['city'].dropna().nunique()
print(f'Raw distinct cities: {before}  ->  cleaned distinct cities: {after}')

Raw distinct cities: 66  ->  cleaned distinct cities: 35


In [13]:
# Audit: list every raw->clean change for transparency
audit = (pd.DataFrame({'raw': df['Name of major city'], 'clean': clean['city']})
         .dropna().drop_duplicates())
changed = audit[audit['raw'].str.strip() != audit['clean']]
print('Distinct city strings altered:', changed['raw'].nunique())
changed.sort_values('clean').head(40)

Distinct city strings altered: 65


,raw,clean
173,"Adilabad, Hanamkonda, Kamareddy, Karimnagar, K...","adilabad, hanamkonda, kamareddy, karimnagar, k..."
133,"Adilabad, Hanamkonda, Kamareddy, Karimnagar, K...","adilabad, hanamkonda, kamareddy, karimnagar, k..."
46,"Adilabad,Hanamkonda,Kamareddy,Karimnagar,Khamm...","adilabad, hanamkonda, kamareddy, karimnagar, k..."
34,Ahemdabad,ahemdabad
41,Ajmer,ajmer
153,"Alwar, Dausa, JCC, JPDC","alwar, dausa, jcc, jpdc"
77,"Alwar, Dausa,Jcc, Jpdc","alwar, dausa, jcc, jpdc"
171,"Anantpur, Kadapa, Kurnool, Nellore, Tirupati","anantpur, kadapa, kurnool, nellore, tirupati"
44,"Anantpur,Kadapa,Kurnool,Nellore,Tirupati","anantpur, kadapa, kurnool, nellore, tirupati"
26,"Badarpur, Karimganj,kokr ajhar, Nagaon, Silcha...","badarpur, karimganj, kokrajhar, nagaon, silcha..."


### 3.4 Validate the measures

In [14]:
# No negatives expected
print('Negative values per measure:')
for c in measure_cols:
    print(f'  {c:14s}: {(clean[c] < 0).sum()}')

# Reconcile total_mu with the two kWh columns (should match within rounding)
m = clean.dropna(subset=['ev_kwh','hd_kwh','total_mu']).copy()
m['calc_mu'] = (m['ev_kwh'] + m['hd_kwh']) / 1_000_000
m['abs_diff'] = (m['calc_mu'] - m['total_mu']).abs()
print('\nMU reconciliation: rows where |calc - reported| > 0.01 :', (m['abs_diff'] > 0.01).sum(), 'of', len(m))

Negative values per measure:
  ev_stations   : 0
  hd_stations   : 0
  ev_kwh        : 0
  hd_kwh        : 0
  total_mu      : 0

MU reconciliation: rows where |calc - reported| > 0.01 : 24 of 570


### 3.5 Recompute `total_mu` for under-reported rows

The reconciliation check flagged **24 rows** where `total_mu` was reported as `0` even though the
detail columns (`ev_kwh` + `hd_kwh`) show real consumption. In these rows the DISCOM filled in the
kilowatt-hour parts but left the megaunit total as a placeholder zero.

The parts are the reliable figures, so we rebuild the total from them:

$$\text{total\_mu} = \frac{\text{ev\_kwh} + \text{hd\_kwh}}{1{,}000{,}000}$$

This is a justified correction, not arbitrary imputation: the correctly-reported rows confirm this
exact relationship holds (1 MU = 1,000,000 kWh), so we apply the dataset's own established formula to
the rows where the total was missing. Only the flagged rows are touched; every already-correct
`total_mu` is left untouched, and before/after values are printed for transparency.

In [15]:
# Recompute total_mu only where it was reported ~0 but the kWh parts are non-zero
parts_present = clean[['ev_kwh', 'hd_kwh']].notna().all(axis=1)
recomputed = (clean['ev_kwh'] + clean['hd_kwh']) / 1_000_000
flag = parts_present & (recomputed > 0.01) & (clean['total_mu'].fillna(0) <= 0.01)

print('Rows being recomputed:', flag.sum())
print('\nBefore -> After (sample):')
print(pd.DataFrame({
    'ev_kwh':       clean.loc[flag, 'ev_kwh'],
    'hd_kwh':       clean.loc[flag, 'hd_kwh'],
    'old_total_mu': clean.loc[flag, 'total_mu'],
    'new_total_mu': recomputed[flag],
}).head(10).to_string(index=False))

clean.loc[flag, 'total_mu'] = recomputed[flag]
print('\nDone. total_mu corrected for', flag.sum(), 'rows.')

Rows being recomputed: 25

Before -> After (sample):
  ev_kwh   hd_kwh  old_total_mu  new_total_mu
430496.0  83625.0          0.00      0.514121
 80946.0 670686.0          0.00      0.751632
 10384.0      0.0          0.01      0.010384
219564.0  19433.0          0.00      0.238997
144612.0  31300.0          0.00      0.175912
 80946.0 670686.0          0.00      0.751632
 80946.0 670686.0          0.00      0.751632
 17127.0      0.0          0.00      0.017127
430496.0  83625.0          0.00      0.514121
430496.0  83625.0          0.00      0.514121

Done. total_mu corrected for 25 rows.


### 3.6 Separate non-reporting rows

Rows blank on **all five** measures are "company filed nothing that month" — not a measured zero. We split them out.

In [16]:
all_blank = clean[measure_cols].isnull().all(axis=1)
print('Non-reporting rows (all measures blank):', all_blank.sum())

non_reporting = clean[all_blank].copy()
analysis = clean[~all_blank].copy()
print('Analysis rows:', len(analysis), '| Non-reporting rows:', len(non_reporting))

Non-reporting rows (all measures blank): 71
Analysis rows: 591 | Non-reporting rows: 71


### 3.7 Quarantine mislabelled rows

Three DISCOMs carry rows whose **city geography is impossible for the labelled state**, and whose
measures **exactly duplicate another DISCOM's legitimate rows** — indicating a corrupted company label
(a data-entry error), not real data:

| Labelled DISCOM (state) | Cities reported | True owner (duplicated) |
|---|---|---|
| `LPDCL` (UT of Ladakh) | Chandigarh, Madhya Pradesh cities | UT Chandigarh / MPMKVVCL |
| `SBPDCL` (Bihar) | Bhubaneswar, Puri… (Odisha) | TPCODL |
| `TSNPDCL` (Telangana) | Coimbatore… (Tamil Nadu) | TANGEDCO |

We cannot reliably recover the true label, and keeping these rows would **double-count** consumption
that already appears under the correct DISCOM. So we **quarantine** them to a separate file and exclude
them from analysis. These are identified by explicit, manually-verified conditions (not an automated
duplicate rule, which would wrongly flag legitimate same-state DISCOMs and cross-border utilities like DVC).

In [17]:
# Explicit, manually-verified mislabel conditions (operate on data-bearing rows in `analysis`)
mis_lpdcl   = analysis['DISCOM'].eq('LPDCL')
mis_sbpdcl  = analysis['DISCOM'].eq('SBPDCL')  & analysis['city'].str.contains('bhubaneswar', case=False, na=False)
mis_tsnpdcl = analysis['DISCOM'].eq('TSNPDCL') & analysis['city'].str.contains('coimbatore', case=False, na=False)
mislabelled_mask = mis_lpdcl | mis_sbpdcl | mis_tsnpdcl

print('Mislabelled rows  -> LPDCL:', mis_lpdcl.sum(),
      '| SBPDCL+Bhubaneswar:', mis_sbpdcl.sum(),
      '| TSNPDCL+Coimbatore:', mis_tsnpdcl.sum())
print('Total quarantined:', mislabelled_mask.sum())

mislabelled = analysis[mislabelled_mask].copy()
analysis = analysis[~mislabelled_mask].copy()
print('Analysis rows after quarantine:', len(analysis))

Mislabelled rows  -> LPDCL: 13 | SBPDCL+Bhubaneswar: 4 | TSNPDCL+Coimbatore: 4
Total quarantined: 21
Analysis rows after quarantine: 570


### 3.8 Final tidy: select, order, and apply case rules

Casing convention: **`city` is fully lowercase, `State` is fully uppercase** (applied to all three output tables for consistency).

In [18]:
final_cols = ['Country', 'financial_year', 'calendar_year', 'calendar_month',
              'DISCOM', 'State', 'city',
              'ev_stations', 'hd_stations', 'ev_kwh', 'hd_kwh', 'total_mu']

# RULE: State column fully uppercase (city is already lowercase from clean_city)
for _df in (analysis, non_reporting, mislabelled):
    _df['State'] = _df['State'].str.upper()

analysis_out = analysis[final_cols].reset_index(drop=True)
non_reporting_out = non_reporting[['Country','financial_year','calendar_year','calendar_month','DISCOM','State','city']].reset_index(drop=True)

print('Final analysis table:', analysis_out.shape)
analysis_out.head()

Final analysis table: (570, 12)


,Country,financial_year,calendar_year,calendar_month,DISCOM,State,city,ev_stations,hd_stations,ev_kwh,hd_kwh,total_mu
0,India,2023-24,2023,8,HESCOM,KAR,hubli,12.0,0.0,12807.0,0.0,0.013
1,India,2023-24,2023,9,HESCOM,KAR,hubli,22.0,0.0,11790.0,0.0,0.012
2,India,2023-24,2023,9,APEPDCL,AP,"visakhapatnam, rajmahendravaram, west godavari...",70.0,0.0,22216.0,0.0,0.022
3,India,2023-24,2023,9,NBPDCL,BIHAR,patna,204.0,0.0,32693.0,0.0,0.033
4,India,2023-24,2023,9,GESCOM,KAR,kalaburgi,1.0,0.0,1498.0,0.0,0.001


In [19]:
# Post-clean summary
print('--- Nulls in final analysis table ---')
print(analysis_out.isnull().sum())
print('\n--- Measure summary ---')
analysis_out[measure_cols].describe()

--- Nulls in final analysis table ---
Country            0
financial_year     0
calendar_year      0
calendar_month     0
DISCOM             8
State              0
city               7
ev_stations       43
hd_stations       17
ev_kwh            17
hd_kwh            17
total_mu           0
dtype: int64

--- Measure summary ---


,ev_stations,hd_stations,ev_kwh,hd_kwh,total_mu
count,527.000000,553.000000,5.530000e+02,5.530000e+02,570.000000
mean,243.574953,3.535262,3.923863e+05,5.544385e+05,0.918581
std,559.548981,5.415283,9.615218e+05,1.389896e+06,1.839645
min,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
25%,11.000000,0.000000,4.013000e+03,0.000000e+00,0.006000
50%,29.000000,1.000000,1.712700e+04,2.732000e+04,0.123500
75%,95.000000,5.000000,1.000540e+05,5.386640e+05,0.782750
max,2782.000000,40.000000,5.252080e+06,1.401472e+07,14.441000


## 4. Export cleaned datasets

In [20]:
analysis_out.to_csv('8486_cleaned_analysis.csv', index=False)
non_reporting_out.to_csv('8486_non_reporting.csv', index=False)

# Mislabelled rows: keep full detail so the quarantine is auditable
mislabelled_out = mislabelled[final_cols].reset_index(drop=True)
mislabelled_out.to_csv('8486_mislabelled_data.csv', index=False)

# A small reference table mapping raw company strings to canonical DISCOM+State
company_map = (pd.DataFrame({'raw': df['Name of distribution company']})
               .drop_duplicates()
               .assign(**{'parsed': lambda d: d['raw']}))
company_map[['DISCOM','State']] = company_map['raw'].apply(split_company)
company_map.loc[company_map['DISCOM']=='SBPDCL','State'] = 'Bihar'
company_map['State'] = company_map['State'].str.upper()
company_map.drop(columns='parsed').to_csv('8486_company_map.csv', index=False)

print('Exported:')
print('  8486_cleaned_analysis.csv ', analysis_out.shape)
print('  8486_non_reporting.csv    ', non_reporting_out.shape)
print('  8486_mislabelled_data.csv ', mislabelled_out.shape)
print('  8486_company_map.csv      ', company_map.shape)

Exported:
  8486_cleaned_analysis.csv  (570, 12)
  8486_non_reporting.csv     (71, 7)
  8486_mislabelled_data.csv  (21, 12)
  8486_company_map.csv       (57, 4)


## 5. Summary

| Step | Result |
|---|---|
| Rows in | 662 |
| Dropped `Additional Info` column | 606/662 null |
| DISCOM spelling variants collapsed | Tata Power-DDL, NPCL (hardcoded) |
| Territory-only entry handled | `UT Chandigarh` → DISCOM=NaN, State=Chandigarh |
| SBPDCL state imputed | 12 rows → Bihar |
| DVC state relabelled | 20 rows → Jharkhand (cross-border) |
| City strings cleaned | case / whitespace / comma / OCR gaps + Kokrajhar fix |
| Footnote removed from city | 1 row |
| `total_mu` recomputed | 24 rows (reported 0, rebuilt from kWh parts) |
| Non-reporting rows split out | 71 |
| Mislabelled rows quarantined | 21 (LPDCL 13, SBPDCL+Bhubaneswar 4, TSNPDCL+Coimbatore 4) |
| **Analysis rows out** | **570** |

**Exports**
- `8486_cleaned_analysis.csv` — analysis-ready, partial nulls kept as NaN.
- `8486_non_reporting.csv` — months a DISCOM filed nothing (71 rows).
- `8486_mislabelled_data.csv` — quarantined label-error rows (21 rows), excluded from analysis.
- `8486_company_map.csv` — raw → canonical company mapping.

These feed the insight-finding notebook next.